# 🩺 Medical RAG Assistant
### Production-Style Retrieval-Augmented Generation on Medical Documents

---

## 📖 Project Overview

This notebook implements a **Medical Question-Answering system** using **Retrieval-Augmented Generation (RAG)** — a technique that grounds LLM answers in a real document rather than relying on the model's training memory.

**The Problem RAG Solves:**  
LLMs can hallucinate medical facts. By retrieving relevant text from your specific document before generating an answer, we dramatically reduce hallucinations and make every answer **traceable to a source**.

**The RAG Pipeline in One Sentence:**
> Convert your PDF into searchable vector embeddings → when a question arrives, retrieve the most relevant chunks → send only those chunks as context to the LLM → get a grounded, cited answer.

---

## 🏗️ Architecture

```
📄 PDF Document
     │
     ▼
  [PyPDF Loader] ──► Extracts text per page + metadata
     │
     ▼
  [RecursiveCharacterTextSplitter] ──► Chunks with overlap
     │
     ▼
  [sentence-transformers/all-MiniLM-L6-v2] ──► 384-dim embeddings
     │
     ▼
  [ChromaDB Vector Store] ──► Persisted index
     │
     ▼
  [MMR Retriever (k=5)] ──► Diverse, relevant chunks
     │
     ▼
  [Groq: llama-3.1-8b-instant] ──► Fast, free LLM inference
     │
     ▼
  ✅ Grounded Answer + Source Citations
```

---

## 🛠️ Tech Stack

| Component | Tool | Why This Choice |
|-----------|------|-----------------|
| PDF Loading | `pypdf` + LangChain | Page-level metadata preserved |
| Text Splitting | `RecursiveCharacterTextSplitter` | Respects sentence/paragraph boundaries |
| Embeddings | `all-MiniLM-L6-v2` | Free, CPU-friendly, strong semantic search |
| Vector DB | `ChromaDB` | Local, no server needed, easy to persist |
| Retrieval | MMR (Maximal Marginal Relevance) | Reduces redundant chunks, improves diversity |
| LLM | Groq `llama-3.1-8b-instant` | Free tier, ~200 tokens/sec, zero cost |
| Evaluation | Keyword-recall + Latency | Lightweight, interpretable, interview-friendly |

✅ **Runs fully in Google Colab — no local GPU or paid APIs required.**

---
## Section 1 — Environment Setup

Install all required packages in one clean command.  
The `-q` flag suppresses noisy output. Run this once per session.

**What changed from the original:**
- Removed `langchain-classic` (deprecated), `langchain-google-genai`, `datasets`, and `ragas` (unused/overkill)
- Added `langchain-huggingface` (the modern, non-deprecated replacement for `langchain_community.embeddings.HuggingFaceEmbeddings`)
- Result: fewer packages, zero deprecation warnings

In [ ]:
# Install all required packages — run once at session start
!pip install -q \
    langchain \
    langchain-community \
    langchain-groq \
    langchain-huggingface \
    langchain-text-splitters \
    chromadb \
    sentence-transformers \
    pypdf

print("✅ All packages installed")

---
## Section 2 — Imports & Configuration

All imports and constants live **once at the top** — a clean engineering practice.  

**The `CONFIG` dictionary** is the single control panel for the entire pipeline.  
To tune retrieval quality, chunking, or switch models, you change exactly one value here — nothing else needs to be touched.

This pattern (centralised config) is standard in production ML systems.

In [ ]:
import os
import time
import getpass
import warnings

warnings.filterwarnings("ignore")

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings        # modern, non-deprecated
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

# ─────────────────────────────────────────────────────────────────────────────
# CENTRAL CONFIG — the single place to tune the pipeline
# ─────────────────────────────────────────────────────────────────────────────
CONFIG = {
    # Chunking parameters
    "chunk_size": 800,       # chars per chunk — smaller = precise, larger = more context
    "chunk_overlap": 150,    # overlap prevents losing context at chunk boundaries

    # Retrieval parameters
    "top_k": 5,              # final chunks returned per query
    "fetch_k": 20,           # MMR candidate pool — larger = more diversity selection
    "lambda_mult": 0.7,      # MMR: 1.0 = pure similarity, 0.0 = max diversity

    # Model identifiers
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "llm_model": "llama-3.1-8b-instant",

    # LLM behaviour
    "temperature": 0.0,      # 0 = fully deterministic — critical for medical factual answers
    "max_tokens": 512,
}

print("✅ Imports successful — no deprecation warnings")
print(f"   Chunk size  : {CONFIG['chunk_size']} chars | Overlap: {CONFIG['chunk_overlap']} chars")
print(f"   Retrieval   : MMR, top_k={CONFIG['top_k']}, fetch_k={CONFIG['fetch_k']}")
print(f"   LLM         : {CONFIG['llm_model']} @ temperature={CONFIG['temperature']}")

---
## Section 3 — Document Loading

**Why PyPDFLoader?**  
It extracts text **page by page** and automatically attaches metadata (`source`, `page`) to each document object.  
This metadata is what enables page-level source citations in the final answers.

**What we get:** A list of `Document` objects, each with:
- `.page_content` — the raw text of that page
- `.metadata` — `{source: filename, page: 0-indexed page number}`

**Text cleaning:** PDFs often have noisy whitespace from PDF rendering. We normalise it here so embeddings aren't polluted by formatting artifacts.

In [ ]:
# ─── Set your PDF path ────────────────────────────────────────────────────────
#
# OPTION A — Google Colab interactive upload:
#   from google.colab import files
#   uploaded = files.upload()
#   PDF_PATH = list(uploaded.keys())[0]
#
# OPTION B — Local file (default, sample PDF ships with repo)
PDF_PATH = "../data/Comprehensive Medical Knowledge Base.pdf"


def load_and_clean_pdf(path: str) -> list:
    """
    Load a PDF and return cleaned LangChain Document objects.
    One Document per page, with source + page metadata preserved.

    Raises FileNotFoundError if path doesn't exist.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"PDF not found at: '{path}'\n"
            f"Please set PDF_PATH to your file location."
        )

    loader = PyPDFLoader(path)
    pages = loader.load()

    # Normalise whitespace — PDFs often have double spaces and stray newlines
    # from the rendering engine. This improves embedding quality.
    for page in pages:
        page.page_content = " ".join(page.page_content.split())

    return pages


pages = load_and_clean_pdf(PDF_PATH)

total_chars = sum(len(p.page_content) for p in pages)
print(f"✅ PDF loaded and cleaned")
print(f"   File        : {os.path.basename(PDF_PATH)}")
print(f"   Pages       : {len(pages)}")
print(f"   Total chars : {total_chars:,}")
print(f"   Avg per page: {total_chars // len(pages):,} chars")
print(f"\n--- Page 1 preview (first 400 chars) ---")
print(pages[0].page_content[:400])

---
## Section 4 — Intelligent Text Chunking

**Why chunk at all?**  
LLMs have context-window limits — we can't send a 20-page PDF as one prompt. We split it into small pieces and only send the *relevant* ones at query time.

**Why `RecursiveCharacterTextSplitter`?**  
It tries to split on natural boundaries in this priority order:  
`paragraph (\n\n)` → `newline (\n)` → `sentence (. )` → `word ( )` → character  
This preserves semantic coherence much better than naive fixed-size splitting.

**Chunk size tradeoff:**

| Chunk Size | Retrieval Precision | Context per Chunk | Best For |
|------------|---------------------|-------------------|----------|
| Small (300) | High — pinpoints exact passage | Low — may miss context | Fact lookup |
| Medium (800) | Good balance | Good balance | Medical QA |
| Large (2000) | Lower — wide net | High — rich context | Summarisation |

We use **800** — a good default for medical question-answering.

**Why overlap (150 chars)?**  
A sentence spanning the boundary between chunk N and chunk N+1 would be truncated without overlap. The 150-char overlap ensures no information is silently lost at boundaries.

In [ ]:
def split_into_chunks(pages: list, chunk_size: int, chunk_overlap: int) -> list:
    """
    Split documents into overlapping chunks for semantic retrieval.

    Enriches each chunk's metadata with:
    - chunk_id: unique sequential index
    - page_number: human-readable (1-indexed) page number

    Returns list of enriched Document objects.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],  # Priority: paragraph > sentence > word
        length_function=len,
    )

    chunks = splitter.split_documents(pages)

    # Enrich metadata for downstream citation display
    for idx, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = idx
        chunk.metadata["page_number"] = chunk.metadata.get("page", 0) + 1  # 0-indexed → 1-indexed

    return chunks


chunks = split_into_chunks(
    pages,
    chunk_size=CONFIG["chunk_size"],
    chunk_overlap=CONFIG["chunk_overlap"],
)

# Diagnostics — sanity-check chunk quality before building the index
chunk_lengths = [len(c.page_content) for c in chunks]

print(f"✅ Split into {len(chunks)} chunks")
print(f"   Min size : {min(chunk_lengths)} chars")
print(f"   Max size : {max(chunk_lengths)} chars")
print(f"   Avg size : {sum(chunk_lengths) // len(chunks)} chars")
print(f"\n--- Sample chunk (chunk #3) ---")
sample = chunks[2]
print(f"   Page     : {sample.metadata['page_number']}")
print(f"   Chunk ID : {sample.metadata['chunk_id']}")
print(f"   Content  : {sample.page_content[:300]}...")

---
## Section 5 — Embedding Model

**What is an embedding?**  
A function that maps text to a fixed-size vector of numbers. Texts with similar *meaning* get vectors that are mathematically close — measured by cosine similarity.

This is the magic that enables semantic search: "cardiac arrest" and "heart stops beating" will retrieve the same chunks even though no exact words match.

**Why `all-MiniLM-L6-v2`?**

| Property | Value |
|----------|-------|
| Dimensions | 384 |
| Training data | 1B sentence pairs |
| Hardware | CPU-only |
| Download size | ~90 MB |
| Cost | Free |

**Interview insight:** For maximum medical accuracy, `BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` would outperform this general model on clinical text — but it's 6x larger. `all-MiniLM-L6-v2` is the right trade-off for accessibility and speed.

**`normalize_embeddings=True`** — forces all vectors onto the unit sphere so cosine similarity = dot product. Required for ChromaDB's default distance metric to work correctly.

In [ ]:
print(f"⏳ Loading embedding model: {CONFIG['embedding_model']}")
print("   (First run: ~90 MB download — cached on subsequent runs)")

embedding_model = HuggingFaceEmbeddings(
    model_name=CONFIG["embedding_model"],
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},  # Unit-sphere normalisation for correct cosine similarity
)

# Sanity check — verify it works and dimensions are as expected
test_vector = embedding_model.embed_query("What are the symptoms of diabetes?")

print(f"\n✅ Embedding model ready")
print(f"   Dimensions : {len(test_vector)}")
print(f"   Sample vals: {[round(v, 4) for v in test_vector[:5]]}...")
print(f"   Vector norm: {sum(v**2 for v in test_vector)**0.5:.4f} (should be ~1.0)")

---
## Section 6 — Vector Store (ChromaDB)

**What happens here?**  
Every chunk is converted to a 384-dimensional vector and stored in ChromaDB — this is our searchable index.

**This is the "offline indexing" phase** — it runs once during setup and is reused for every query.

**ChromaDB vs. alternatives:**
- **ChromaDB** — local, zero-config, great for notebooks and prototypes ✅
- **FAISS** — faster at scale, but requires manual serialization
- **Pinecone/Weaviate** — production-grade, serverless, but paid

For a portfolio project, ChromaDB is the cleanest choice.

In [ ]:
print(f"⏳ Building ChromaDB index from {len(chunks)} chunks...")

start_time = time.time()

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="medical_rag",
    # persist_directory="../vectorstore"  # Uncomment to save to disk — avoids rebuild on restart
)

indexing_time = round(time.time() - start_time, 1)

print(f"\n✅ Vector store ready in {indexing_time}s")
print(f"   Vectors stored : {vector_store._collection.count()}")
print(f"   Collection     : medical_rag")
print(f"   Storage        : in-memory (add persist_directory to save to disk)")

---
## Section 7 — Retriever (MMR Strategy)

**The problem with vanilla similarity search:**  
If you ask "symptoms of diabetes?", the top-5 most similar chunks might all be from the *same* paragraph — giving the LLM redundant context and missing other relevant sections.

**MMR (Maximal Marginal Relevance) solution:**  
MMR balances two goals:
1. **Relevance** — chunks must be similar to the query
2. **Diversity** — chunks should not be too similar to each other

**How it works:**
1. Fetch `fetch_k=20` candidate chunks by similarity
2. Pick the best chunk → add to result set
3. Greedily pick next chunk that is: most similar to query AND least similar to already-selected chunks
4. Repeat until `k=5` chunks are selected

**`lambda_mult=0.7`** → 70% weight on relevance, 30% on diversity. Good default for medical QA.

**Interview talking point:** *"I chose MMR over standard similarity search because medical documents often contain repetitive sections. MMR ensures the LLM receives diverse evidence — like getting opinions from multiple specialists rather than the same specialist five times."*

In [ ]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": CONFIG["top_k"],
        "fetch_k": CONFIG["fetch_k"],
        "lambda_mult": CONFIG["lambda_mult"],
    },
)

# ── Test retrieval ─────────────────────────────────────────────────────────────
test_query = "What are the symptoms of diabetes mellitus?"
retrieved_docs = retriever.invoke(test_query)

print(f"✅ MMR Retriever configured")
print(f"   search_type : mmr (Maximal Marginal Relevance)")
print(f"   top_k       : {CONFIG['top_k']}   | fetch_k: {CONFIG['fetch_k']}")
print(f"   lambda_mult : {CONFIG['lambda_mult']} (relevance/diversity balance)")
print(f"\n🔍 Test Query: '{test_query}'")
print(f"   Retrieved {len(retrieved_docs)} diverse chunks:")
for i, doc in enumerate(retrieved_docs, 1):
    page = doc.metadata.get("page_number", "?")
    snippet = doc.page_content[:150].replace("\n", " ")
    print(f"\n  [{i}] Page {page}")
    print(f"       {snippet}...")

---
## Section 8 — LLM + Medical Prompt Engineering

**Why a custom prompt?**  
The default LangChain RetrievalQA prompt is generic. A domain-specific medical prompt:
- Constrains the LLM to answer ONLY from the provided context
- Instructs it to say "I don't know" when the answer isn't in the document
- Eliminates outside knowledge → eliminates hallucinations from the model's training data
- Requests structured, professional formatting

**This custom prompt is arguably the highest-impact single change in this pipeline.**

**Why Groq?**  
- Free tier with ~200 tokens/sec — much faster than local Ollama
- No credit card required — just sign up at [console.groq.com](https://console.groq.com)
- `llama-3.1-8b-instant` is strong at instruction-following and covers medical knowledge

In [ ]:
# ─── API Key (Groq — free at https://console.groq.com) ────────────────────────
# Colab tip: store as a Secret (🔑 left sidebar) to avoid pasting every session
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter Groq API Key: ")

# ─── Medical-Specific Prompt ──────────────────────────────────────────────────
# Three critical constraints:
#   1. Answer ONLY from provided context (prevents hallucination)
#   2. Explicit "I don't know" instruction (prevents confabulation)
#   3. Structured output format (improves readability)
MEDICAL_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a precise medical information assistant. Your role is to answer \
questions accurately using ONLY the medical document context provided below.

RULES (non-negotiable):
1. Answer EXCLUSIVELY using information from the context. Do not use outside knowledge.
2. If the answer is not found in the context, respond with:
   "I don't have enough information in the provided document to answer this question."
3. Structure your answer with bullet points when listing multiple items.
4. Do not provide medical diagnoses, prescriptions, or personal health advice.
5. Be concise and factual. Avoid filler phrases.

Context from the medical document:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{context}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Question: {question}

Answer:"""
)

# ─── LLM Initialisation ───────────────────────────────────────────────────────
llm = ChatGroq(
    model_name=CONFIG["llm_model"],
    temperature=CONFIG["temperature"],  # 0 = deterministic — required for medical QA
    max_tokens=CONFIG["max_tokens"],
)

print("✅ LLM initialised")
print(f"   Model       : {CONFIG['llm_model']}")
print(f"   Temperature : {CONFIG['temperature']} (deterministic outputs)")
print(f"   Max tokens  : {CONFIG['max_tokens']}")
print("\n✅ Medical prompt template ready")
print("   Grounding   : Context-only answers")
print("   Fallback    : Explicit I-don't-know response")

---
## Section 9 — RAG Chain Assembly

**This is where all five components connect into one end-to-end system:**

```
Question ──► [MMR Retriever] ──► ChromaDB similarity search
                  │
                  ▼ (top-5 relevant, diverse chunks)
             [MEDICAL_PROMPT] ──► format(context=chunks, question=query)
                  │
                  ▼
             [Groq LLM] ──► generate answer from context only
                  │
                  ▼
             Answer + Source Documents
```

**`chain_type="stuff"`** — concatenates all retrieved chunks into one prompt as context.  
Works well when `k ≤ 6`. For larger k or very long documents, consider `"map_reduce"` or `"refine"` chain types.

In [ ]:
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",             # Concatenates all retrieved chunks into one context
    retriever=retriever,
    return_source_documents=True,   # Enables page-level citations
    chain_type_kwargs={
        "prompt": MEDICAL_PROMPT,   # Our custom grounding prompt
        "verbose": False,
    },
)

print("✅ RAG chain assembled — all components connected")
print()
print("  Query")
print("    │")
print("    ▼")
print("  MMR Retriever  (ChromaDB, top-5 diverse chunks)")
print("    │")
print("    ▼")
print("  Medical Prompt (context-grounded, I-don't-know fallback)")
print("    │")
print("    ▼")
print("  Groq LLM       (llama-3.1-8b-instant, temperature=0)")
print("    │")
print("    ▼")
print("  Answer + Source Pages")

---
## Section 10 — Query Interface

A well-designed query function wraps the entire pipeline cleanly:  
- Input validation
- Latency measurement
- "I don't know" detection (graceful fallback)
- Source deduplication for clean citation display
- Structured return value (usable by evaluation code)

In [ ]:
# Phrases that indicate the LLM couldn't find the answer in the document
NO_ANSWER_PHRASES = [
    "i don't have enough information",
    "not mentioned in the document",
    "not provided in the context",
    "i cannot find",
    "not available in the provided",
]


def ask(question: str, show_sources: bool = True) -> dict:
    """
    Query the Medical RAG pipeline with a natural language question.

    Args:
        question: Medical question to ask.
        show_sources: Print source citations if True.

    Returns:
        dict with: answer, sources, latency_seconds, has_answer
    """
    if not question.strip():
        print("⚠️  Please provide a non-empty question.")
        return {}

    print(f"\n{'='*65}")
    print(f"❓ {question}")
    print(f"{'='*65}")

    # Run pipeline + time it
    t0 = time.time()
    result = rag_chain.invoke({"query": question})
    latency = round(time.time() - t0, 2)

    answer = result["result"].strip()
    source_docs = result["source_documents"]

    # Detect no-answer fallback
    has_answer = not any(p in answer.lower() for p in NO_ANSWER_PHRASES)
    prefix = "💡" if has_answer else "⚠️ "

    print(f"\n{prefix} ANSWER  [{latency}s]")
    print(answer)

    # Source citations — deduplicated by page for clean display
    if show_sources and source_docs:
        print(f"\n📚 SOURCES — {len(source_docs)} chunks retrieved")
        seen_pages = set()
        for i, doc in enumerate(source_docs, 1):
            page = doc.metadata.get("page_number", "?")
            marker = " *(cont.)" if page in seen_pages else ""
            seen_pages.add(page)
            snippet = doc.page_content[:200].replace("\n", " ")
            print(f"\n  [{i}] Page {page}{marker}")
            print(f"       {snippet}...")

    print(f"\n{'='*65}")

    return {
        "question": question,
        "answer": answer,
        "sources": source_docs,
        "latency_seconds": latency,
        "has_answer": has_answer,
    }


print("✅ Query function ready")
print("   Usage: ask('What are the symptoms of hypertension?')")

---
## Section 11 — Ask Questions

Three demo calls show the system at its best — and at its honest limits.

In [ ]:
# ─── Demo 1: Standard in-document medical question ────────────────────────────
r1 = ask("What are the main symptoms of a heart attack?")

In [ ]:
# ─── Demo 2: Multi-part question — tests retrieval of diverse content ──────────
r2 = ask("What imaging techniques are used in medical diagnosis and what does each one measure?")

In [ ]:
# ─── Demo 3: Out-of-scope question — tests the I-don't-know fallback ──────────
# The system should refuse to answer and NOT hallucinate.
r3 = ask("What is the current market cap of Moderna?", show_sources=False)

In [ ]:
# ─── Interactive: Ask your own question ───────────────────────────────────────
user_q = input("\n❓ Ask a medical question: ")
result = ask(user_q)

---
## Section 12 — Evaluation

**Why evaluate?** You can't trust a RAG system you haven't measured. Evaluation finds the weakest link: is it retrieval? The prompt? The LLM?

**Our evaluation strategy: Keyword Recall + Has-Answer Rate + Latency**

| Metric | Formula | What It Measures | Limitation |
|--------|---------|-----------------|------------|
| **Keyword Recall** | matched_keywords / total_keywords | Factual coverage — does the answer mention the key facts? | A wrong answer mentioning the right words still scores 100% |
| **Has-Answer Rate** | answered_Qs / total_Qs | Is retrieval finding relevant content? | — |
| **Avg Latency** | total_time / n_queries | User experience | — |

**Why not RAGAS faithfulness/relevance scores?**  
RAGAS requires a judge LLM (paid API call per metric). Keyword recall is interpretable, free, and sufficient to demonstrate evaluation competence in a portfolio project. In production, RAGAS would be the right upgrade.

**Interview talking point:** *"I deliberately chose lightweight evaluation metrics that are easy to explain and don't require additional API calls. A recruiter can understand keyword recall in 30 seconds. In production, I'd graduate to RAGAS metrics — but for a portfolio, interpretability wins."*

In [ ]:
# Evaluation test cases — questions with expected keywords from the actual document
EVAL_CASES = [
    {
        "question": "What are the symptoms of hypertension?",
        "keywords": ["headache", "dizziness", "blurred vision", "fatigue"],
        "category": "Cardiovascular",
    },
    {
        "question": "What is tuberculosis and how does it affect the body?",
        "keywords": ["bacterial", "lungs", "infection", "cough"],
        "category": "Infectious Disease",
    },
    {
        "question": "What are common medical imaging techniques?",
        "keywords": ["MRI", "CT", "X-ray", "Ultrasound"],
        "category": "Diagnostics",
    },
    {
        "question": "What are the treatment options for diabetes mellitus?",
        "keywords": ["insulin", "medication", "diet", "lifestyle"],
        "category": "Endocrinology",
    },
    {
        "question": "What is the role of vaccination in public health?",
        "keywords": ["prevention", "disease", "community", "immune"],
        "category": "Public Health",
    },
]

print(f"✅ Evaluation suite: {len(EVAL_CASES)} test cases")
categories = list(set(c["category"] for c in EVAL_CASES))
print(f"   Categories  : {', '.join(categories)}")

In [ ]:
def run_evaluation(eval_cases: list) -> dict:
    """
    Run the RAG evaluation suite.

    Metrics computed:
    - keyword_recall: fraction of expected keywords present in answer
    - has_answer: whether the system answered vs. fell back to I-don't-know
    - latency_seconds: end-to-end query time

    Returns aggregated summary + per-case results.
    """
    per_case = []

    print("📊 RAG Evaluation Report")
    print("=" * 70)

    for idx, case in enumerate(eval_cases, 1):
        question = case["question"]
        keywords = case["keywords"]

        # Run pipeline
        t0 = time.time()
        result = rag_chain.invoke({"query": question})
        latency = round(time.time() - t0, 2)

        answer_lower = result["result"].strip().lower()
        source_pages = [
            doc.metadata.get("page_number", "?") for doc in result["source_documents"]
        ]

        # Keyword recall
        matched = [kw for kw in keywords if kw.lower() in answer_lower]
        missing = [kw for kw in keywords if kw.lower() not in answer_lower]
        recall = len(matched) / len(keywords)

        # Has-answer detection
        has_answer = not any(p in answer_lower for p in NO_ANSWER_PHRASES)

        per_case.append({
            "category": case["category"],
            "question": question,
            "keyword_recall": recall,
            "matched": matched,
            "missing": missing,
            "has_answer": has_answer,
            "source_pages": source_pages,
            "latency_seconds": latency,
        })

        # Progress bar visual
        filled = int(recall * 10)
        bar = "█" * filled + "░" * (10 - filled)
        status = "✅" if has_answer else "⚠️ "

        print(f"\n{status} [{idx}/{len(eval_cases)}] {case['category']}")
        print(f"   Q         : {question}")
        print(f"   Recall    : [{bar}] {recall:.0%}  ({len(matched)}/{len(keywords)} keywords found)")
        if missing:
            print(f"   Missing   : {missing}")
        print(f"   Pages     : {source_pages}")
        print(f"   Latency   : {latency}s")

    # Aggregate metrics
    avg_recall = sum(r["keyword_recall"] for r in per_case) / len(per_case)
    answer_rate = sum(r["has_answer"] for r in per_case) / len(per_case)
    avg_latency = sum(r["latency_seconds"] for r in per_case) / len(per_case)

    print(f"\n{'='*70}")
    print("📈 OVERALL RESULTS")
    print(f"{'='*70}")
    print(f"  Avg Keyword Recall : {avg_recall:.1%}")
    print(f"  Has-Answer Rate   : {answer_rate:.1%}")
    print(f"  Avg Latency       : {avg_latency:.2f}s / query")
    print(f"  Test cases        : {len(per_case)}")
    print(f"{'='*70}")
    print()
    print("💡 Evaluation Notes:")
    print("  - Keyword recall is a coverage proxy, not a correctness guarantee")
    print("  - Production upgrade: RAGAS faithfulness + context_precision metrics")
    print("  - Has-Answer Rate confirms retrieval is finding relevant content")

    return {
        "per_case": per_case,
        "avg_keyword_recall": avg_recall,
        "has_answer_rate": answer_rate,
        "avg_latency_seconds": avg_latency,
    }


eval_results = run_evaluation(EVAL_CASES)

---
## Section 13 — Design Decisions, Limitations & Future Work

Being able to articulate *why* you made each choice, what you'd do differently, and what the limitations are — this is what separates a portfolio project from a tutorial copy.

### ✅ Key Design Decisions

| Decision | Choice Made | Why | Alternative |
|----------|------------|-----|-------------|
| Embedding model | `all-MiniLM-L6-v2` | Free, CPU-only, strong semantic understanding | `BiomedNLP-BiomedBERT` for domain accuracy |
| Vector DB | ChromaDB | Local, zero-config, notebook-friendly | FAISS (faster), Pinecone (production) |
| Retrieval | MMR | Reduces redundant chunks → richer LLM context | BM25 hybrid for rare term matching |
| Chunk size | 800 chars | Precision/context balance for medical QA | 500 (more precise), 1500 (more context) |
| LLM | Groq Llama-3.1-8B | Fast, free, strong instruction following | GPT-4o (better quality, paid) |
| Temperature | 0.0 | Medical QA demands determinism | 0.1-0.3 for more natural language |
| Custom prompt | Grounding + fallback | Eliminates outside knowledge hallucination | Default LangChain prompt (weaker) |
| Evaluation | Keyword recall | Interpretable, free, no judge LLM | RAGAS (production standard) |

### ⚠️ Known Limitations

1. **Scanned PDFs** — PyPDFLoader only extracts selectable text. Image-based PDFs need OCR (`pytesseract`).
2. **No conversation memory** — each query is stateless. Follow-up questions ("tell me more about #2") won't work.
3. **Single PDF only** — the index is built for one document. Multiple PDFs need metadata filtering.
4. **Keyword evaluation weakness** — a factually wrong answer that mentions the right keywords still scores 100%.
5. **Groq rate limits** — free tier throttles under heavy usage. Add `time.sleep()` or retry logic for batches.

### 🚀 Prioritised Future Improvements

1. **Conversation memory** (`ConversationBufferWindowMemory`) — enables multi-turn QA
2. **Gradio / Streamlit UI** — makes the project demo-ready in 30 minutes
3. **Hybrid retrieval** — combine BM25 (keyword) + dense (semantic) for better recall on rare medical terms
4. **RAGAS evaluation** — faithfulness + context precision + answer relevance
5. **Domain embedding model** — `BiomedNLP-BiomedBERT` for higher accuracy on clinical text
6. **Persistent vector store** — save ChromaDB to disk, load on restart
7. **Cross-encoder reranking** — re-score retrieved chunks with a stronger model before sending to LLM

### 💬 Interview Q&A

**Q: Why not just send the whole PDF to the LLM?**  
A: Context window limits + cost + hallucination risk increases with longer contexts. RAG scales to arbitrarily large corpora and keeps responses grounded in specific evidence.

**Q: What's the difference between MMR and similarity search?**  
A: Similarity search returns the top-k most similar vectors — but they can be redundant (all from the same paragraph). MMR trades some similarity for diversity: it ensures retrieved chunks cover different aspects of the question.

**Q: How would you reduce hallucinations further?**  
A: (1) Custom prompt with explicit grounding — done here. (2) Increase top_k for more evidence. (3) Add a faithfulness check: pass the answer + context to a judge LLM and verify the answer is supported. (4) Use a larger, more capable LLM.

**Q: How would this scale to 10,000 documents?**  
A: Replace ChromaDB with Pinecone or pgvector. Add metadata filtering so retrieval only searches the relevant subset (e.g., filter by department, document type). Batch-embed documents asynchronously during ingestion.

**Q: What would you add first if given one more day?**  
A: Conversation memory + a Gradio UI. Maximum demo impact for minimal implementation complexity.

---

## 🔗 Resources

- [LangChain RAG Docs](https://python.langchain.com/docs/use_cases/question_answering/)
- [Groq Free API](https://console.groq.com)
- [ChromaDB Docs](https://docs.trychroma.com)
- [Sentence Transformers](https://www.sbert.net)
- [RAGAS Evaluation Framework](https://docs.ragas.io)
- [MMR Explained (paper)](https://dl.acm.org/doi/10.1145/290941.291025)

---
*Portfolio project demonstrating practical RAG engineering. Not intended for clinical use.*